# Exercise 12 — Market efficiency

Homework, about 30 minutes, any tools allowed (AI included). Three short tasks on one
century of S&P 500 returns: are returns predictable from their own past, from the
dividend yield, and would you have trusted the answer in real time?

In the student version three code cells contain a `TODO`; everything else runs as it is.
Each task ends with one question — write two or three sentences per answer. There is no
debrief in class: the solution notebook is posted on Monday 7 December so you can check
your answers.

Data: Amit Goyal and Ivo Welch, updated predictor database through December 2025
([sites.google.com/view/agoyal145](https://sites.google.com/view/agoyal145)), January 1926
to December 2025.

Start with **Runtime → Restart and run all**.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf

from exercise_utils import FHNW, setup_style
setup_style()

In [ ]:
URL = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
       "Exercise_12/data/Data2025.xlsx")

raw = pd.read_excel(URL, sheet_name="Monthly")
raw.index = pd.PeriodIndex(raw["yyyymm"].astype(str), freq="M")
raw = raw.loc["1926-01":"2025-12"]

ret = raw["ret"].astype(float)                                   # monthly total return
lrx = np.log(1 + ret) - np.log(1 + raw["Rfree"].astype(float))   # log excess return
dp = (raw["d12"] / raw["price"]).astype(float)                   # dividend-price ratio

print(f"{len(ret)} months, {ret.index[0]} to {ret.index[-1]}")

## Task 1 — Do past returns predict next month's return?

Weak-form efficiency says returns are close to unpredictable from their own past. Compute
the autocorrelations $\rho_1, \dots, \rho_{12}$ of the monthly return. The figure marks the
95 % band $\pm 1.96/\sqrt{T}$.

**Question.** Is $\rho_1$ statistically significant? What share of the variance of next
month's return does it explain ($\rho_1^2$)?

In [ ]:
T = len(ret)
band = 1.96 / np.sqrt(T)
# TODO: rho = autocorrelations of ret at lags 1 to 12 (acf from statsmodels, drop lag 0)
print(f"rho(1) = {rho[0]:+.4f}   band = +/-{band:.4f}   rho(1)^2 = {rho[0] ** 2:.2%}")

In [ ]:
fig1, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(np.arange(1, 13), rho, color=FHNW["blue"], width=0.6)
for s in (band, -band):
    ax.axhline(s, color=FHNW["red"], lw=1.0, ls="--")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(np.arange(1, 13))
ax.set_xlabel("lag, months")
ax.set_ylabel("autocorrelation")
plt.show()

## Task 2 — Does the dividend yield predict long-horizon returns?

Regress the log excess return over the next $h$ months on today's dividend-price ratio,

$$\sum_{j=1}^{h} r_{t+j} \;=\; a + b \cdot (D_t/P_t) + \varepsilon_{t+h},$$

for $h$ = 1, 12 and 60 months. For $h > 1$ consecutive observations overlap — two
five-year returns starting one month apart share 59 months — so OLS standard errors are
far too small. Newey-West (HAC) standard errors correct for this:
`.fit(cov_type="HAC", cov_kwds={"maxlags": h})`.

**Question.** From one month to five years, $R^2$ and t(OLS) rise steeply. How many
non-overlapping five-year windows does the century contain, and which t-statistic should
you believe?

In [ ]:
# regress the sum of y over t+1 ... t+h on x at date t
def predictive(y, x, h):
    df = pd.concat([y.rolling(h).sum().shift(-h).rename("y"), x.rename("x")], axis=1).dropna()
    X = sm.add_constant(df["x"])
    # TODO: fit the regression twice: plain OLS (call it ols) and with Newey-West errors (nw)
    return {"b": ols.params["x"], "t(OLS)": ols.tvalues["x"], "t(NW)": nw.tvalues["x"],
            "R2": ols.rsquared, "N": len(df), "windows": len(df) // h}

In [ ]:
tab = pd.DataFrame({f"h = {h}": predictive(lrx, dp, h) for h in (1, 12, 60)}).T
tab[["N", "windows"]] = tab[["N", "windows"]].astype(int)
print(tab.to_string(float_format=lambda v: f"{v:8.3f}"))

## Task 3 — Would you have believed it in 1975?

Run the five-year regression ($h = 60$) separately on the two halves of the century,
1926–1975 and 1976–2025. The figure shows both halves.

**Question.** An investor in 1975 sees only the first half. What would she conclude, and
what did the next fifty years show?

In [ ]:
HALVES = [("1926-1975", "1926-01", "1975-12"), ("1976-2025", "1976-01", "2025-12")]
# TODO: run predictive() at h = 60 on each half and collect the rows in a DataFrame called halves
halves["windows"] = halves["windows"].astype(int)
print(halves[["b", "t(NW)", "R2", "windows"]].to_string(float_format=lambda v: f"{v:8.3f}"))

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, (name, a, b) in zip(axes, HALVES):
    y5 = lrx.loc[a:b].rolling(60).sum().shift(-60)
    df = pd.concat([y5.rename("y"), dp.loc[a:b].rename("x")], axis=1).dropna()
    slope, intercept = np.polyfit(df["x"], df["y"], 1)
    grid = np.linspace(df["x"].min(), df["x"].max(), 50)
    ax.scatter(df["x"], df["y"], s=5, color=FHNW["blue"], alpha=0.4)
    ax.plot(grid, intercept + slope * grid, color=FHNW["red"], lw=1.8)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("D/P at date t")
axes[0].set_ylabel("log excess return, next 5 years")
plt.show()

### Your answers

Write two or three sentences for each of the three questions. The solution notebook is
posted on Monday 7 December. Drill 12 is in Moodle and closes on Sunday 6 December at 23:59.